# 3 - Sélection du meilleur modèle via MLflow

## Objectif

Utiliser **MLflow Client** pour :
1. Rechercher tous les runs loggés
2. Sélectionner le meilleur modèle selon le F1-score
3. Charger le modèle avec `mlflow.pyfunc.load_model`
4. Le tester sur le dataset

## Workflow

```
1. Connexion MLflow + MlflowClient
2. Recherche des runs (search_runs)
3. Comparaison et sélection du meilleur
4. Chargement du modèle (load_model)
5. Évaluation sur le test set
```

---

## 1. Configuration et imports

In [1]:
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

from functions.data_load import load_dataset
from functions.data_preparation import prepare_data
from ml_config import MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT_TUNING

RANDOM_STATE = 42
METRIC = "f1"

print("Imports OK")

d:\Suivi_de_formation\simplon-ai-developer-training\W31-W33-ML-DEPLOYMENT-CI-CD\mlflow-EDUCATIONAL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


d:\Suivi_de_formation\simplon-ai-developer-training\W31-W33-ML-DEPLOYMENT-CI-CD\mlflow-EDUCATIONAL\.venv\Lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


---

## 2. Connexion MLflow et recherche des runs

In [2]:
# Connexion
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

# Récupérer l'experiment
experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT_TUNING)
if experiment is None:
    raise ValueError(f"Experiment '{MLFLOW_EXPERIMENT_TUNING}' introuvable sur {MLFLOW_TRACKING_URI}")

print(f"Experiment: {experiment.name} (id: {experiment.experiment_id})")
print(f"Tracking URI: {MLFLOW_TRACKING_URI}")

Experiment: tuning (id: 3)
Tracking URI: http://localhost:5000


In [3]:
# Rechercher tous les runs terminés, triés par F1 décroissant
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="status = 'FINISHED'",
    order_by=[f"metrics.{METRIC} DESC"],
)

print(f"{len(runs)} runs trouvés\n")

# Tableau récapitulatif
runs_data = []
for run in runs:
    runs_data.append({
        "run_id": run.info.run_id[:8],
        "run_name": run.info.run_name,
        "model_type": run.data.tags.get("model_type", "?"),
        "f1": run.data.metrics.get("f1"),
        "accuracy": run.data.metrics.get("accuracy"),
        "precision": run.data.metrics.get("precision"),
        "recall": run.data.metrics.get("recall"),
        "auc": run.data.metrics.get("auc"),
    })

df_runs = pd.DataFrame(runs_data)
df_runs

239 runs trouvés



,run_id,run_name,model_type,f1,accuracy,precision,recall,auc
0,ca1dabae,tuning_CatBoost 25/02/2026 11h:46m:31s,CatBoost,0.6877,0.6904,0.6859,0.6904,0.7091
1,5fd698b3,tuning_CatBoost 24/02/2026 16h:03m:57s,CatBoost,0.6877,0.6904,0.6859,0.6904,0.7091
2,703b9d12,tuning_XGBoost 25/02/2026 11h:45m:53s,XGBoost,0.6872,0.6890,0.6857,0.6890,0.7093
3,aeb73217,tuning_XGBoost 24/02/2026 16h:03m:18s,XGBoost,0.6872,0.6890,0.6857,0.6890,0.7093
4,0736bfdc,tuning_XGB_baseline 25/02/2026 11h:45m:19s,XGB_baseline,0.6858,0.6874,0.6844,0.6874,0.7087
...,...,...,...,...,...,...,...,...
234,47321b10,hyperopt_randomforest_#005,randomforest,NaN,NaN,NaN,NaN,NaN
235,68012c21,hyperopt_randomforest_#004,randomforest,NaN,NaN,NaN,NaN,NaN
236,8a39aace,hyperopt_randomforest_#003,randomforest,NaN,NaN,NaN,NaN,NaN
237,bb6a768d,hyperopt_randomforest_#002,randomforest,NaN,NaN,NaN,NaN,NaN


---

## 3. Sélection du meilleur modèle

In [4]:
# Le premier run est le meilleur (tri DESC)
best_run = runs[0]
best_run_id = best_run.info.run_id
best_run_name = best_run.info.run_name
best_f1 = best_run.data.metrics.get("f1")
best_model_type = best_run.data.tags.get("model_type", "inconnu")

print("=" * 50)
print("MEILLEUR RUN")
print("=" * 50)
print(f"Run ID    : {best_run_id}")
print(f"Nom       : {best_run_name}")
print(f"Modèle    : {best_model_type}")
print(f"F1        : {best_f1:.4f}")
print(f"Accuracy  : {best_run.data.metrics.get('accuracy', 0):.4f}")
print(f"Precision : {best_run.data.metrics.get('precision', 0):.4f}")
print(f"Recall    : {best_run.data.metrics.get('recall', 0):.4f}")
print(f"AUC       : {best_run.data.metrics.get('auc', 0):.4f}")

MEILLEUR RUN
Run ID    : ca1dabaea8464788a7e2ded831c3e16f
Nom       : tuning_CatBoost 25/02/2026 11h:46m:31s
Modèle    : CatBoost
F1        : 0.6877
Accuracy  : 0.6904
Precision : 0.6859
Recall    : 0.6904
AUC       : 0.7091


---

## 4. Chargement du modèle depuis MLflow

On liste les artefacts du run pour trouver le chemin du modèle,
puis on le charge avec `mlflow.pyfunc.load_model`.

In [5]:
# En MLflow 3.x, les modèles ne sont plus dans les artefacts classiques.
# Il faut utiliser search_logged_models() pour les trouver.
logged_models = mlflow.search_logged_models(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"source_run_id = '{best_run_id}'",
    output_format="list",
)

if not logged_models:
    raise FileNotFoundError(f"Aucun modèle loggé trouvé pour le run {best_run_id[:8]}")

# Afficher les modèles loggés pour ce run
print(f"Modèles loggés pour le run {best_run_id[:8]} :")
for lm in logged_models:
    print(f"  - name: {lm.name}")
    print(f"    model_id: {lm.model_id}")
    print(f"    source_run_id: {lm.source_run_id}")

# Prendre le premier modèle loggé
best_logged_model = logged_models[0]
print(f"\nModèle sélectionné : {best_logged_model.name}")

Modèles loggés pour le run ca1dabae :
  - name: tuning-CatBoost 25-02-2026 11h46m33s
    model_id: m-606776a9d5d44bbabd04ffbc51ad08c7
    source_run_id: ca1dabaea8464788a7e2ded831c3e16f

Modèle sélectionné : tuning-CatBoost 25-02-2026 11h46m33s


In [6]:
# Charger le modèle via son URI MLflow 3.x
# Deux options : models:/{model_id} ou runs:/{run_id}/{name}
model_uri = f"models:/{best_logged_model.model_id}"
print(f"Chargement depuis : {model_uri}")

loaded_model = mlflow.pyfunc.load_model(model_uri)
print(f"Modèle chargé : {type(loaded_model)}")

Chargement depuis : models:/m-606776a9d5d44bbabd04ffbc51ad08c7


Modèle chargé : <class 'mlflow.pyfunc.PyFuncModel'>


---

## 5. Évaluation sur le dataset

In [7]:
# Charger et préparer les données
df_accident = load_dataset("dataset_accident")
X, y = prepare_data(df_accident, "grav_binary")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.9, random_state=RANDOM_STATE, stratify=y
)

print(f"Test set : {len(X_test)} échantillons")
print(f"Features : {list(X_test.columns)}")

dataset_accident: chargé depuis Parquet (263,356 lignes, 11 colonnes)
Test set : 237021 échantillons
Features : ['est_nuit', 'est_heure_pointe', 'jour_semaine', 'est_weekend', 'agg', 'vma', 'impl_vehicule_leger', 'impl_poids_lourd', 'impl_pieton']


In [8]:
# Prédictions avec le modèle chargé depuis MLflow
y_pred = loaded_model.predict(X_test)

# Métriques
print("=" * 50)
print(f"EVALUATION - {best_model_type} (chargé depuis MLflow)")
print("=" * 50)
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")

print(f"\nClassification Report :")
print(classification_report(y_test, y_pred, target_names=["Non grave", "Grave"]))

print("Matrice de confusion :")
print(confusion_matrix(y_test, y_pred))

EVALUATION - CatBoost (chargé depuis MLflow)
Accuracy  : 0.4773
Precision : 0.5514
Recall    : 0.4773
F1-Score  : 0.4838

Classification Report :
              precision    recall  f1-score   support

   Non grave       0.66      0.41      0.50    153764
       Grave       0.36      0.61      0.45     83257

    accuracy                           0.48    237021
   macro avg       0.51      0.51      0.48    237021
weighted avg       0.55      0.48      0.48    237021

Matrice de confusion :
[[62530 91234]
 [32663 50594]]


In [9]:
# Vérification : F1 loggé vs F1 recalculé
f1_recalc = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"F1 loggé dans MLflow : {best_f1:.4f}")
print(f"F1 recalculé ici     : {f1_recalc:.4f}")

if abs(best_f1 - f1_recalc) < 0.01:
    print("\nLes scores correspondent - le modèle est correctement chargé.")
else:
    print(f"\nEcart de {abs(best_f1 - f1_recalc):.4f}")

F1 loggé dans MLflow : 0.6877
F1 recalculé ici     : 0.4838

Ecart de 0.2039
